# Browser History Domain Overview

This notebook gives a first rough overview of your browser history by grouping visits per domain.


## 0) Upload your browser history CSV

**In JupyterLite (browser):** Drag and drop your `BrowserHistory_*.csv` file onto the
file browser in the left sidebar, then update `CSV_FILENAME` below.

**Locally:** Place the CSV file in the same directory as this notebook and update `CSV_FILENAME`.

> Export from Edge: `edge://history` → ... → Export browsing history

In [ ]:
# ── Configure your CSV filename here ──────────────────────────────────────
CSV_FILENAME = "BrowserHistory_22.02.26.csv"
# ──────────────────────────────────────────────────────────────────────────

# In JupyterLite: use the file browser (left sidebar) to upload your CSV first,
# then re-run this cell with the correct CSV_FILENAME.
try:
    import pyodide  # noqa: F401  – we're in JupyterLite
    import ipywidgets as widgets
    import io, js

    uploader = widgets.FileUpload(accept='.csv', multiple=False)
    out = widgets.Output()

    def on_upload(change):
        with out:
            out.clear_output()
            if uploader.value:
                fname, meta = next(iter(uploader.value.items()))
                content = meta['content']
                with open(fname, 'wb') as f:
                    f.write(bytes(content))
                global CSV_FILENAME
                CSV_FILENAME = fname
                print(f"✅ Uploaded: {fname} ({len(content):,} bytes)")
                print("Now run the next cell to load the data.")

    uploader.observe(on_upload, names='value')
    display(widgets.VBox([
        widgets.Label(f'Current file: {CSV_FILENAME}'),
        widgets.Label('Or upload a new CSV:'),
        uploader,
        out
    ]))
except ImportError:
    # Running locally — just use the filename as-is
    print(f"Running locally. Using: {CSV_FILENAME}")


In [ ]:
from pathlib import Path
from initialisation import load_history_df

csv_path = Path(CSV_FILENAME)  # set in cell above

# Load CSV with pandas, parse DateTime as index, and add domain column
df = load_history_df(csv_path)

print(f"Loaded {len(df):,} history rows from {csv_path.name}")
print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nIndex range: {df.index.min()} to {df.index.max()}")


## 2) Extract domain from each URL
Normalizes domains by lowering case and stripping a leading `www.`.

In [ ]:
# `domain` is already prepared by load_history_df() from initialisation.py
domain_counts = df["domain"].value_counts()

print(f"Unique domains: {len(domain_counts):,}")
print(f"Total visits: {domain_counts.sum():,}")

## 3) Top domains
Shows the highest-frequency domains for a rough overview.

In [ ]:
top_n = 25
top_domains = domain_counts.head(top_n)

print(f"Top {top_n} domains by visit count:\n")
for idx, (domain, count) in enumerate(top_domains.items(), start=1):
    print(f"{idx:>2}. {domain:<35} {count:>8,}")

## 4) Optional quick buckets
Creates coarse buckets to reduce noise from many one-off domains.

In [ ]:
bucketed = pd.Series(index=[">=100", "25-99", "5-24", "1-4"], dtype=int).fillna(0)

for count in domain_counts.values:
    if count >= 100:
        bucket = ">=100"
    elif count >= 25:
        bucket = "25-99"
    elif count >= 5:
        bucket = "5-24"
    else:
        bucket = "1-4"
    bucketed[bucket] += 1

for label in [">=100", "25-99", "5-24", "1-4"]:
    print(f"Domains with {label:>5} visits: {bucketed[label]:,}")

In [ ]:
from urllib.parse import urlparse, parse_qs

def is_google_search(url: str) -> bool:
    """Check if URL is a Google search"""
    if not url:
        return False
    parsed = urlparse(url)
    return parsed.netloc.lower() in ("www.google.com", "google.com", "google.de", "google.co.uk")

def extract_search_query(url: str) -> str:
    """Extract search query from Google URL"""
    if not url:
        return None
    parsed = urlparse(url)
    params = parse_qs(parsed.query)
    q = params.get("q", [None])[0]
    return q.replace("+", " ") if q else None

def has_ai_overview(url: str) -> bool:
    """Check if search used Google AI Overview feature (udm=50)"""
    if not url:
        return False
    parsed = urlparse(url)
    params = parse_qs(parsed.query)
    udm = params.get("udm", [None])[0]
    return udm == "50"

# Apply detection functions to the DataFrame
df["is_google_search"] = df["NavigatedToUrl"].apply(is_google_search)
df["search_query"] = df["NavigatedToUrl"].apply(lambda url: extract_search_query(url) if is_google_search(url) else None)
df["has_ai_overview"] = df["NavigatedToUrl"].apply(has_ai_overview)

# Stats
total_google_searches = df["is_google_search"].sum()
ai_searches = df["has_ai_overview"].sum()

print(f"Total Google searches: {total_google_searches:,}")
print(f"AI-powered searches (udm=50): {ai_searches:,}")
if total_google_searches > 0:
    pct = (ai_searches / total_google_searches) * 100
    print(f"Percentage using AI Overview: {pct:.1f}%")

## 6) Top AI Search Queries
Shows the most frequent queries using AI Overview feature.

In [ ]:
ai_queries = df[df["has_ai_overview"]]["search_query"].value_counts().head(15)

print(f"Top 15 AI-assisted search queries:\n")
for idx, (query, count) in enumerate(ai_queries.items(), start=1):
    if query:
        print(f"{idx:>2}. {query:<60} {count:>4,}")

## 7) All Google searches by topic
Groups all Google search queries (both AI-powered and traditional) by topic category and counts them.

In [ ]:
def classify_search_topic(query: str) -> str:
    """Classify a search query into a topic category"""
    if not query:
        return "Other"
    
    query_lower = query.lower()
    
    # Define topic keywords
    topics = {
        "AI/ML": ["copilot", "agent", "gpt", "llm", "neural", "ml", "machine learning", 
                  "deep learning", "ai", "chatgpt", "ort", "whisper", "mcp"],
        "Programming": ["python", "javascript", "react", "code", "debug", "node", "typescript", 
                        "function", "error", "api", "sdk", "library", "nodejs", "npm", "github"],
        "Hardware/Electronics": ["arduino", "raspberry", "circuit", "usb", "microcontroller", 
                                 "fnirsi", "multimeter", "instek", "breadboard", "sensor"],
        "Automotive/Vehicles": ["mercedes", "volvo", "car", "bmw", "automobile", "engine", 
                                "motor", "vehicle", "driving"],
        "Shopping/Ecommerce": ["amazon", "ebay", "shop", "buy", "price", "product", "sale", 
                               "aliexpress", "store"],
        "News/Current": ["news", "breaking", "update", "announced", "release", "launched"],
        "Tools/Utilities": ["tool", "utility", "software", "app", "application", "extension"],
        "General Info": ["what", "how", "why", "explain", "definition", "meaning", "vs", "vs."],
    }
    
    # Check each topic's keywords
    for topic, keywords in topics.items():
        for keyword in keywords:
            if keyword in query_lower:
                return topic
    
    return "Other"

# Extract all Google searches (not just AI-powered)
all_google_searches = df[df["is_google_search"]]["search_query"].dropna()

# Apply topic classification
topics = all_google_searches.apply(classify_search_topic)

# Count by topic
topic_counts = topics.value_counts().sort_values(ascending=False)

print(f"Search topics (all {len(all_google_searches):,} Google searches):\n")
for topic, count in topic_counts.items():
    pct = (count / len(all_google_searches)) * 100
    print(f"{topic:<20} {count:>6,} ({pct:>5.1f}%)")

## 8) Top searches by topic
Shows the most frequent queries in each topic category.

In [ ]:
# Create a DataFrame with queries and topics for better grouping
query_df = pd.DataFrame({
    "query": all_google_searches.values,
    "topic": topics.values
})

# Show top 5 queries per topic (excluding "Other" for readability)
for topic in topic_counts.index:
    if topic == "Other":
        continue  # Skip "Other" as it's too broad
    
    top_queries = query_df[query_df["topic"] == topic]["query"].value_counts().head(5)
    if len(top_queries) > 0:
        print(f"\n{topic}:")
        for query, count in top_queries.items():
            print(f"  • {query:<55} ({count})")

# Show sample of "Other" category
print(f"\nOther (sample of {topic_counts['Other']:,} queries):")
other_queries = query_df[query_df["topic"] == "Other"]["query"].value_counts().head(10)
for query, count in other_queries.items():
    print(f"  • {query:<55} ({count})")

## 9) Google visits without search queries
Breaks down the 617 Google visits that don't have an extractable search query.

In [ ]:
# Find Google visits that are NOT search queries
google_visits = df[df["is_google_search"]]
no_query_visits = google_visits[google_visits["search_query"].isna()]

print(f"Total Google.com visits:        {len(google_visits):,}")
print(f"Google searches (with query):   {len(all_google_searches):,}")
print(f"Google visits without query:    {len(no_query_visits):,}")
print(f"Difference:                     {len(google_visits) - len(all_google_searches):,}\n")

# Analyze non-search page titles to see what they were
print("Sample page titles from non-search Google visits:")
non_search_titles = no_query_visits["PageTitle"].dropna().value_counts().head(15)
for title, count in non_search_titles.items():
    print(f"  • {title:<60} ({count})")

## 10) Google search usage over time
Analyzes search query volume by day and week to show usage patterns and trends.

In [ ]:
# Get Google searches grouped by day
google_searches_daily = df[df["is_google_search"] & df["search_query"].notna()].copy()

# Group by date (day) and count queries
queries_per_day = google_searches_daily.groupby(google_searches_daily.index.date).size()

# Statistics
print(f"Google Search Usage Over Time")
print(f"{'='*50}")
print(f"Total days with searches: {len(queries_per_day):,}")
print(f"Date range: {queries_per_day.index.min()} to {queries_per_day.index.max()}")
print(f"\nDaily query statistics:")
print(f"  Average per day: {queries_per_day.mean():.1f}")
print(f"  Median per day:  {queries_per_day.median():.0f}")
print(f"  Max in a day:    {queries_per_day.max()}")
print(f"  Min in a day:    {queries_per_day.min()}")

# Show top 15 days
print(f"\nTop 15 days by search volume:")
top_days = queries_per_day.nlargest(15)
for i, (date, count) in enumerate(top_days.items(), start=1):
    print(f"{i:>2}. {date}  {count:>4} searches")

# Weekly aggregation to show trend
queries_per_week = google_searches_daily.resample("W").size()

print(f"\n\nWeekly search aggregation:")
print(f"{'='*50}")
print(f"Week Ending          Queries  Avg/Day")
print(f"{'-'*50}")
for week_end, count in queries_per_week.items():
    avg_per_day = count / 7
    print(f"{week_end.date()}     {count:>6}    {avg_per_day:>6.1f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ------------------------------------------------------------------
# 11) All  visits over time
all_visits = df.copy()

# daily / weekly counts
visits_per_day = all_visits.groupby(all_visits.index.date).size()
visits_per_week = all_visits.resample("W").size()

print("Non‑Google visits over time")
print("="*50)
print(f"Days tracked: {len(visits_per_day):,}")
print(f"Date range: {visits_per_day.index.min()} to {visits_per_day.index.max()}")
print(f"\nDaily stats: avg {visits_per_day.mean():.1f}, max {visits_per_day.max()}, min {visits_per_day.min()}")

print("\nWeekly aggregation")
print("-"*50)
print("Week Ending          Visits  Avg/Day")
for week_end, cnt in visits_per_week.items():
    print(f"{week_end.date()}     {cnt:>6}    {cnt/7:>6.1f}")

# plot it with the same style as before
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.plot(visits_per_day.index, visits_per_day.values,
         linewidth=1.5, color="purple", marker="o", markersize=3, alpha=0.7)
ax1.set_title("Daily  Visits", fontsize=14, fontweight="bold")
ax1.set_ylabel("Visits", fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax1.axhline(visits_per_day.mean(), color="red", linestyle="--",
            linewidth=2, label=f"Average: {visits_per_day.mean():.1f}", alpha=0.7)
ax1.legend(fontsize=10)

ax2.bar(range(len(visits_per_week)), visits_per_week.values,
        width=0.6, color="olive", alpha=0.7, edgecolor="black")
ax2.set_title("Weekly  Visits", fontsize=14, fontweight="bold")
ax2.set_ylabel("Visits", fontsize=11)
ax2.set_xlabel("Week", fontsize=11)
ax2.grid(True, alpha=0.3, axis="y")
ax2.set_xticks(range(len(visits_per_week)))
ax2.set_xticklabels([d.strftime("%m-%d") for d in visits_per_week.index],
                    rotation=45, ha="right")
ax2.axhline(visits_per_week.mean(), color="red", linestyle="--",
            linewidth=2, label=f"Weekly Avg: {visits_per_week.mean():.0f}", alpha=0.7)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Daily rows: {len(visits_per_day)}  weeks: {len(visits_per_week)}")

## 12+) Topic classification moved
The extracted topic/classification section now lives in `browser_history_topic_overview.ipynb`.
This notebook now focuses on the domain and Google-search overview sections.